In [52]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [53]:
import numpy as np
import numpy.linalg as LA
import pandas as pd
import jax
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist

jax.config.update("jax_enable_x64", True)

from mixres.sim import PoissonEnveloped1D, make_aggregation_matrix
from mixres.models._inference import run_inference_mcmc

from bspline1d_utils import (
    make_bspline_basis,
    make_gmrf_precision,
    gmrf_covariance_cholesky,
)
from interval_utils import (
    expand_interval_dataframe,
    compute_interval_widths,
    build_interval_sum_matrix,
)

# For visualization
import altair as alt

## Utility functions

In [54]:
def summarize_interval_segments(df_segment: pd.DataFrame) -> pd.DataFrame:
    return (
        df_segment[["obs_interval", "x", "lambda"]]
        .drop_duplicates(["obs_interval", "x"])
        .groupby("obs_interval", as_index=False)
        .agg(x_min=("x", "min"), x_max=("x", "max"), lambda_obs=("lambda", "mean"))
    )


def plot_data(
    df_obs: pd.DataFrame,
    df_rates: pd.DataFrame,
    show_empirical: bool = True,
    show_legend: bool = True,
    color_scale: alt.Scale | None = None,
) -> alt.LayerChart:
    """
    Plot the true rates, the average rates over the observed intervals, and optionally the empirical rates from the observed data.

    Parameters
    ----------
    df_obs : pd.DataFrame
        DataFrame containing the observed data with columns 'obs_interval', 'x', and 'y'.
    df_rates : pd.DataFrame
        DataFrame containing the true rates with columns 'x' and 'lambda'.
    show_empirical : bool, optional
        If True, include the empirical rates in the plot. Default is True.
    show_legend : bool, optional
        If True, show a legend labeling each series. Default is True.
    color_scale : alt.Scale | None, optional
        Color scale to use for the "series" field, keyed by "True rate",
        "Interval average", and (if `show_empirical`) "Empirical rate". If
        None (default), a scale covering just those series is built here.
        Pass an explicit scale when layering this chart's output with other
        charts that share the "series" color field -- e.g. `plot_posterior`
        -- since Vega-Lite only keeps the *first* layer's `range` when
        merging per-layer color scales that declare different `range`
        arrays, silently reusing colors for domain values added by later
        layers otherwise.

    Returns
    -------
    alt.LayerChart
                An Altair chart object containing the plot.

    See Also
    --------
    PoissonEnveloped1D
    """

    df_segment = expand_interval_dataframe(
        df_obs, interval_col="obs_interval", x_col="x"
    )
    source_segment = summarize_interval_segments(df_segment)

    if color_scale is None:
        legend_domain = ["True rate", "Interval average"]
        legend_range = ["#de425b", "#594e90"]
        if show_empirical:
            legend_domain.append("Empirical rate")
            legend_range.append("#ffa600")
        color_scale = alt.Scale(domain=legend_domain, range=legend_range)
    color_legend = alt.Legend(title=None) if show_legend else None

    chart_true = (
        alt.Chart(df_rates.assign(series="True rate"))
        .mark_line()
        .encode(
            x=alt.X("x:Q", title="x"),
            y=alt.Y("lambda:Q", title="λ"),
            color=alt.Color(
                "series:N", scale=color_scale, title=None, legend=color_legend
            ),
        )
    )

    chart_segment = (
        alt.Chart(source_segment.assign(series="Interval average"))
        .mark_rule(strokeWidth=2.5, opacity=0.85)
        .encode(
            x=alt.X("x_min:Q", title="x"),
            x2="x_max:Q",
            y=alt.Y("lambda_obs:Q", title="λ"),
            color=alt.Color(
                "series:N", scale=color_scale, title=None, legend=color_legend
            ),
        )
    )

    if show_empirical:
        df_emp = df_obs.groupby("obs_interval")["y"].agg(["sum", "count"]).reset_index()
        df_emp["lambda"] = df_emp["sum"] / df_emp["count"]
        df_emp = expand_interval_dataframe(
            df_emp, interval_col="obs_interval", x_col="x"
        )
        source_emp = summarize_interval_segments(df_emp)

        chart_emp = (
            alt.Chart(source_emp.assign(series="Empirical rate"))
            .mark_rule(strokeWidth=2.5, opacity=0.85)
            .encode(
                x=alt.X("x_min:Q", title="x"),
                x2="x_max:Q",
                y=alt.Y("lambda_obs:Q", title="λ"),
                color=alt.Color(
                    "series:N", scale=color_scale, title=None, legend=color_legend
                ),
            )
        )

        return chart_true + chart_segment + chart_emp
    else:
        return chart_true + chart_segment


def make_fisher_information(
    lambda_: np.ndarray, C: np.ndarray, Phi: np.ndarray, K: np.ndarray | None = None
) -> np.ndarray:
    """
    Computes the Fisher information matrix for the Poisson model given the rate,
    aggregation matrix, and B-spline basis.

    If `K` is given (the Cholesky factor of a coefficient covariance prior,
    Sigma = K @ K.T), the Fisher information is reparametrized into the
    Whitened coordinates beta = K @ alpha, alpha ~ N(0, I): I_alpha = K.T @
    I_beta @ K. This lets the identifiability analysis (eigendecomposition
    into image/null space) reflect a spatial smoothness prior instead of
    treating coefficient directions as independent.

    Parameters
    ----------
    lambda_ : np.ndarray
        The Poisson rate parameter for each observation.
    C : np.ndarray
        The composition matrix mapping fine-grained rates to observed intervals.
    Phi : np.ndarray
        The B-spline basis matrix.
    K : np.ndarray | None, optional
        The Cholesky factor of a coefficient covariance prior, Sigma = K @ K.T.
    """

    inv_eta = 1 / (C @ lambda_ / C.sum(axis=1))
    Dinv_eta = np.diag(inv_eta)
    Dlam = np.diag(lambda_)

    I = Phi.T @ Dlam @ C.T @ Dinv_eta @ C @ Dlam @ Phi
    if K is not None:
        I = K.T @ I @ K
    return I

## Setup Block

In [55]:
# Input space
x = np.linspace(0, 100, 100)

# Number of B-spline basis functions
M = 40

# Intervals
# Size 10 intervals
# left_limits = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90]
# right_limits = [9, 19, 29, 39, 49, 59, 69, 79, 89, 99]

# Size 20 intervals
# left_limits = [0, 20, 40, 60, 80]
# right_limits = [19, 39, 59, 79, 99]

left_limits = [0, 10, 20, 30, 50, 70, 80, 90]
right_limits = [9, 19, 29, 49, 69, 79, 89, 99]

# left_limits = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90]
# right_limits = [9, 19, 29, 39, 49, 59, 69, 79, 89, 99]

### Generate Data

In [56]:
intervals = [f"[{left},{right}]" for left, right in zip(left_limits, right_limits)]
print("Intervals:", intervals)

df_obs, df_rates = PoissonEnveloped1D(
    obs_intervals=intervals, n_observations=1000, seed=1
).generate(return_true_rates=True)

Intervals: ['[0,9]', '[10,19]', '[20,29]', '[30,49]', '[50,69]', '[70,79]', '[80,89]', '[90,99]']


Visualise the data.

In [57]:
plot_data(df_obs, df_rates, show_empirical=True)

alt.LayerChart(...)

### Generate Key Objects

In [58]:
C = make_aggregation_matrix(left_limits, right_limits)
Phi = make_bspline_basis(x=x, n_basis=M)
Q = make_gmrf_precision(Phi.shape[1], order=2, tau=1.0, jitter=1e-4)
K = gmrf_covariance_cholesky(Q)

## Subspace Decomposition

### Setting GMRF-Regularized Fisher Information

Settings 1-3 all treat coefficient directions as a priori independent: the eigendecomposition of `Phi.T @ Dlam @ W.T @ Dinv_eta @ W @ Dlam @ Phi` splits `beta`-space into image/null subspaces with no notion that neighboring spline coefficients should be spatially correlated. This setting instead places a Gaussian Markov Random Field (GMRF) smoothness prior on `beta`, expressed via its covariance's Cholesky factor `K` (`Sigma = K @ K.T`, from `make_gmrf_precision` / `gmrf_covariance_cholesky` in `bspline1d_utils`), and folds it into the Fisher information as `K.T @ I @ K` -- the Fisher information of the whitened coordinates `alpha` under the reparametrization `beta = K @ alpha`, `alpha ~ N(0, I)`.

We reuse the empirical point estimate `lambda_hat` from Setting 2, so this setting differs from Setting 2 only in reparametrizing coefficient space by the GMRF prior before eigendecomposing -- not in how `lambda_` (and hence `Dlam`) is built. `PrIm_reg`/`PrKer_reg` therefore live in `alpha`-space, not raw `beta`-space: reconstructing `f` requires `Phi @ K @ (...)` rather than `Phi @ (...)`.

In [59]:
df_eta_hat = df_obs.groupby("obs_interval_id")["y"].agg(N="size", y="sum").reset_index()
eta_hat = (df_eta_hat["y"] / df_eta_hat["N"]).to_numpy()

interval_avg_op = C / C.sum(axis=1, keepdims=True)
lambda_hat = np.linalg.pinv(interval_avg_op) @ eta_hat
assert np.allclose(interval_avg_op @ lambda_hat, eta_hat)

In [60]:
I = make_fisher_information(lambda_hat, C, Phi, K=K)

eigvals, eigvecs = LA.eigh(I)
tol = eigvals.max() * len(eigvals) * np.finfo(eigvals.dtype).eps
PrIm = eigvecs[:, eigvals > tol]
PrKer = eigvecs[:, eigvals <= tol]

print(PrIm.shape, PrKer.shape)

(40, 8) (40, 32)


In [61]:
def plot_prior_subspace_samples(
    PrIm: np.ndarray,
    PrKer: np.ndarray,
    Phi: np.ndarray,
    K: np.ndarray,
    n_samples: int = 1,
    seed: int | None = None,
    y_min: float | None = None,
    y_max: float | None = None,
) -> alt.Chart:
    """
    Plot standardized prior draws of the identifiable and non-identifiable
    function components.

    For each of `n_samples` draws, standard normal coefficients are mapped
    through `PrIm` (and `PrKer`, when non-trivial) and `Phi @ K` onto the
    evaluation grid, standardized, and overlaid as semi-transparent curves --
    visualizing how much prior variation the identifiable subspace of the
    (GMRF-regularized) Fisher information admits relative to its null space.

    Parameters
    ----------
    PrIm : np.ndarray, shape (M, r)
        Basis for the image (identifiable subspace) of the Fisher information.
    PrKer : np.ndarray, shape (M, M - r)
        Basis for the null space (non-identifiable subspace) of the Fisher
        information. If `PrKer.shape[1] == 0` (full-rank Fisher information),
        no non-identifiable curves are drawn.
    Phi : np.ndarray, shape (N, M)
        B-spline basis matrix.
    K : np.ndarray, shape (M, M)
        Cholesky factor of the coefficient covariance prior (Sigma = K @ K.T).
    n_samples : int, optional
        Number of prior draws to overlay per subspace. Default is 1.
    seed : int | None, optional
        Seed for the random number generator, for reproducible draws.
        Default is None.
    y_min : float | None, optional
        Lower limit of the y-axis. Curve segments below this value are
        clipped rather than shown. If None (default), it is chosen
        automatically from the data.
    y_max : float | None, optional
        Upper limit of the y-axis. Curve segments above this value are
        clipped rather than shown. If None (default), it is chosen
        automatically from the data.

    Returns
    -------
    alt.Chart
        Overlaid, semi-transparent prior-draw curves, colored by subspace
        ("Identifiable" / "Non-identifiable").
    """
    rng = np.random.default_rng(seed)
    x_grid = np.linspace(0, 100, Phi.shape[0])

    def draw_samples(Pr: np.ndarray, label: str) -> pd.DataFrame:
        alpha = rng.normal(0, 1, size=(Pr.shape[1], n_samples))
        f = Phi @ K @ (Pr @ alpha)
        f = (f - f.mean(axis=0)) / f.std(axis=0)
        return pd.DataFrame(
            {
                "x": np.tile(x_grid, n_samples),
                "eta": f.T.flatten(),
                "sample": np.repeat(np.arange(n_samples), len(x_grid)),
                "series": label,
            }
        )

    frames = [draw_samples(PrIm, "Identifiable")]
    legend_domain = ["Identifiable"]
    legend_range = ["#7aa609"]

    if PrKer.shape[1] > 0:
        frames.append(draw_samples(PrKer, "Non-identifiable"))
        legend_domain.append("Non-identifiable")
        legend_range.append("#de425b")

    source = pd.concat(frames, ignore_index=True)

    y_scale_kwargs = {}
    if y_min is not None:
        y_scale_kwargs["domainMin"] = y_min
    if y_max is not None:
        y_scale_kwargs["domainMax"] = y_max

    return (
        alt.Chart(source)
        .mark_line(opacity=0.7, clip=True)
        .encode(
            x=alt.X("x:Q", title="x"),
            y=alt.Y(
                "eta:Q",
                title="η (standardized)",
                scale=alt.Scale(**y_scale_kwargs),
            ),
            color=alt.Color(
                "series:N",
                scale=alt.Scale(domain=legend_domain, range=legend_range),
                title=None,
            ),
            detail="sample:N",
        )
    )


plot_prior_subspace_samples(
    PrIm, PrKer, Phi, K, n_samples=3, seed=42, y_min=-5, y_max=5
)

alt.Chart(...)

## Result Plotting Helpers

Shared helpers for the 4 downstream experiments' result plots, so each experiment's own plot and the final side-by-side comparison are built from the same code.

In [62]:
def summarize_posterior_lam(
    mcmc_samples: dict, x: np.ndarray, label: str
) -> pd.DataFrame:
    """Posterior mean and 95% band for `lam`, tagged with an experiment label."""
    central = mcmc_samples["lam"].mean(axis=0)
    lower = np.percentile(mcmc_samples["lam"], 2.5, axis=0)
    upper = np.percentile(mcmc_samples["lam"], 97.5, axis=0)
    return pd.DataFrame(
        {
            "x": x,
            "lam": central.flatten(),
            "lower": lower.flatten(),
            "upper": upper.flatten(),
            "experiment": label,
        }
    )


def plot_posterior(
    source: pd.DataFrame,
    df_obs: pd.DataFrame,
    df_rates: pd.DataFrame,
    title: str | None = None,
) -> alt.LayerChart:
    """Posterior mean + 95% band, layered over the true curve and raw interval markers."""
    color_scale = alt.Scale(
        domain=["True rate", "Interval average", "Empirical rate", "Posterior mean"],
        range=["#de425b", "#594e90", "#ffa600", "purple"],
    )
    chart = (
        alt.Chart(source.assign(series="Posterior mean"))
        .mark_line()
        .encode(
            x="x",
            y="lam",
            color=alt.Color("series:N", scale=color_scale, title=None),
        )
    )
    band = (
        alt.Chart(source)
        .mark_area(color="purple", opacity=0.2)
        .encode(x="x", y="lower", y2="upper")
    )
    result = plot_data(df_obs, df_rates, color_scale=color_scale) + chart + band
    return result.properties(title=title) if title else result

In [63]:
def summarize_posterior_lam_betaIm_median(
    mcmc_samples: dict,
    PrIm: ArrayLike,
    Phi_eff: ArrayLike,
    x: np.ndarray,
    label: str,
) -> pd.DataFrame:
    """
    Posterior median of `lam` computed from `betaIm` only (the identified
    subspace), with the uncertainty band taken from the full posterior `lam`
    -- which comes from jointly fitting `betaIm` and `betaKer` via MCMC, so
    the null-space contribution to the band is data-informed rather than a
    raw prior draw. (A prior-only draw of `betaKer` produced implausibly
    wide bands, suggesting the null space isn't actually as uninformed as
    the Fisher-information analysis assumed -- jointly fitting it lets the
    likelihood constrain it instead.)
    """
    beta0 = mcmc_samples["beta0"]
    betaIm = mcmc_samples["betaIm"]

    lam_Im = jnp.exp(beta0[:, None] + (Phi_eff @ (PrIm @ betaIm.T)).T)
    central = np.median(lam_Im, axis=0)

    lam_full = mcmc_samples["lam"]
    lower = np.percentile(lam_full, 2.5, axis=0)
    upper = np.percentile(lam_full, 97.5, axis=0)

    return pd.DataFrame(
        {
            "x": x,
            "lam": np.asarray(central).flatten(),
            "lower": np.asarray(lower).flatten(),
            "upper": np.asarray(upper).flatten(),
            "experiment": label,
        }
    )

In [64]:
def summarize_posterior_lam_nullspace_prior_uncertainty(
    mcmc_samples: dict,
    PrIm: ArrayLike,
    PrKer: ArrayLike,
    Phi_eff: ArrayLike,
    x: np.ndarray,
    label: str,
    rng_key: ArrayLike,
    scale: float = 1.0,
) -> pd.DataFrame:
    """
    Posterior median of `lam` from betaIm only (`mcmc_samples["lam"]`, from a
    model_reduced_bspline run with no betaKer), with an uncertainty band
    built by drawing one fresh betaKer ~ Normal(0, I) per posterior betaIm
    sample and pushing it through Phi_eff -- i.e. null-space uncertainty
    taken from its prior, not fit to data.
    """
    beta0 = mcmc_samples["beta0"]
    betaIm = mcmc_samples["betaIm"]
    S = beta0.shape[0]
    MKer = PrKer.shape[1]

    betaKer_prior = jax.random.normal(rng_key, shape=(S, MKer)) * scale
    lam_full = jnp.exp(
        beta0[:, None] + (Phi_eff @ (PrIm @ betaIm.T + PrKer @ betaKer_prior.T)).T
    )

    central = np.median(mcmc_samples["lam"], axis=0)
    lower = np.percentile(lam_full, 2.5, axis=0)
    upper = np.percentile(lam_full, 97.5, axis=0)

    return pd.DataFrame(
        {
            "x": x,
            "lam": np.asarray(central).flatten(),
            "lower": np.asarray(lower).flatten(),
            "upper": np.asarray(upper).flatten(),
            "experiment": label,
        }
    )

### Experiment 1: Naive B-Spline model

In [65]:
def model_bspline(
    C: jax.Array,
    Phi: jax.Array,
    N: jax.Array,
    id: jax.Array,
    K: jax.Array | None = None,
    y=None,
):
    """
    Naive Bayesian B-spline regression model.

    Parameters
    ----------
    C : jax.Array
        Composition matrix mapping fine-scale rates to observed intervals.
    Phi : jax.Array
        B-spline basis matrix.
    N : jax.Array
        Number of observations in each interval.
    id : jax.Array
        Interval indices for each observation.
    K : jax.Array | None, optional
        Cholesky factor of the coefficient covariance prior (Sigma = K @ K.T).
    y : jax.Array, optional
        Observed counts in each interval. If None, the model will generate synthetic data.
    """
    M = Phi.shape[1]

    # --- Priors ---
    beta0 = numpyro.sample("beta0", dist.Normal(0, 1))
    beta = numpyro.sample("w", dist.Normal(0, 1).expand([M]).to_event(1))
    Phi_eff = Phi @ K if K is not None else Phi

    # --- Latent rate ---
    lam = numpyro.deterministic("lam", jnp.exp(beta0 + Phi_eff @ beta))
    lam_interval = (
        jnp.matmul(C, lam) / C.sum(axis=1) * N[id]
    )  # Average rate over each interval (divide by interval-specific width)

    # --- Likelihood ---
    with numpyro.plate("obs", len(y)):
        numpyro.sample("y", dist.Poisson(lam_interval[id]), obs=y)

In [66]:
# Prepare the data for the model
df_model = df_obs.groupby(["obs_interval_id"])["y"].agg(N="size", y="sum").reset_index()

# Convert to JAX arrays
model_kwargs = {
    "C": jnp.array(C),
    "Phi": jnp.array(Phi),
    "N": jnp.array(df_model["N"].values),
    "id": jnp.array(df_model["obs_interval_id"].values),
    "K": jnp.array(K),
    "y": jnp.array(df_model["y"].values),
}

In [67]:
# MCMC inference with NUTS
rng_key_mcmc = jax.random.PRNGKey(0)
mcmc = run_inference_mcmc(rng_key_mcmc, model_bspline, **model_kwargs)
mcmc_samples = mcmc.get_samples()

/Users/shozen/Imperial/0_Research/mixres/mixres/models/_inference.py:28: UserWarning: There are not enough devices to run parallel chains: expected 4 but got 1. Chains will be drawn sequentially. If you are running MCMC in CPU, consider using `numpyro.set_host_device_count(4)` at the beginning of your program. You can double-check how many devices are available in your system using `jax.local_device_count()`.
  mcmc = MCMC(
sample: 100%|██████████| 1000/1000 [00:04<00:00, 243.31it/s, 1023 steps of size 8.27e-04. acc. prob=0.94]

Number of divergences: 0


In [68]:
source_exp1 = summarize_posterior_lam(
    mcmc_samples, np.linspace(0, 100, 100), "1. Naive"
)
plot_posterior(source_exp1, df_obs, df_rates)

alt.LayerChart(...)

### Experiment 2: Reduced subspace model

In [ ]:
def model_reduced_bspline(
    W: ArrayLike,
    Phi: ArrayLike,
    PrIm: ArrayLike,
    N: ArrayLike,
    interval_id: ArrayLike,
    y=None,
    K: ArrayLike | None = None,
):
    """
    Naive Bayesian B-spline regression model.

    Parameters
    ----------
    W : ArrayLike
        Aggregation matrix mapping fine-scale rates to observed intervals.
    Phi: ArrayLike
        B-spline basis matrix.
    PrIm: ArrayLike
        Projection matrix onto the image of the Fisher information matrix.
    N: ArrayLike
        Number of observations in each interval.
    interval_id: ArrayLike
        Interval indices for each observation.
    y: ArrayLike, optional
        Observed counts in each interval. If None, the model will generate synthetic data.
    K: ArrayLike, optional
        Cholesky factor of a coefficient covariance prior (Sigma = K @ K.T),
        e.g. a GMRF smoothness prior. If given, `PrIm`/`PrKer` are assumed to
        live in the whitened coordinates beta = K @ alpha, so the basis is
        effectively Phi @ K instead of Phi. If None (default), no
        reparametrization is applied.
    """
    M = PrIm.shape[1]
    Phi_eff = Phi @ K if K is not None else Phi

    # --- Priors ---
    beta0 = numpyro.sample("beta0", dist.Normal(0, 1))
    betaIm = numpyro.sample("betaIm", dist.Normal(0, 1).expand([M]).to_event(1))

    # --- Latent rate ---
    lam = numpyro.deterministic("lam", jnp.exp(beta0 + Phi_eff @ (PrIm @ betaIm)))
    lam_interval = jnp.matmul(W, lam) / W.sum(axis=1) * N[interval_id]

    # --- Likelihood ---
    with numpyro.plate("obs", len(y)):
        numpyro.sample("y", dist.Poisson(lam_interval[interval_id]), obs=y)

In [ ]:
PrIm_jax = jnp.array(PrIm)

In [ ]:
# MCMC inference with NUTS
rng_key_mcmc = jax.random.PRNGKey(1)
mcmc = run_inference_mcmc(
    rng_key_mcmc,
    model_reduced_bspline,
    W=W_jax,
    Phi=Phi_jax,
    PrIm=PrIm_jax,
    N=N,
    interval_id=interval_id,
    K=K_jax,
    y=y_jax,
)
mcmc_samples = mcmc.get_samples()
print(mcmc_samples.keys())

In [ ]:
source_exp2 = summarize_posterior_lam(
    mcmc_samples, np.linspace(0, 100, 100), "2. Reduced Subspace"
)
plot_posterior(source_exp2, df_obs, df_rates)

### Experiment 3: Null-space included model

In [ ]:
def model_nullinc_bspline(
    W: ArrayLike,
    Phi: ArrayLike,
    PrIm: ArrayLike,
    PrKer: ArrayLike,
    N: ArrayLike,
    interval_id: ArrayLike,
    interval_width: ArrayLike,
    y=None,
    K: ArrayLike | None = None,
):
    """
    Naive Bayesian B-spline regression model.

    Parameters
    ----------
    W : ArrayLike
        Aggregation matrix mapping fine-scale rates to observed intervals.
    Phi: ArrayLike
        B-spline basis matrix.
    PrIm: ArrayLike
        Projection matrix onto the image of the Fisher information matrix.
    N: ArrayLike
        Number of observations in each interval.
    inverval_id: ArrayLike
        Interval indices for each observation.
    interval_width: ArrayLike
        Width of each interval.
    y: ArrayLike, optional
        Observed counts in each interval. If None, the model will generate synthetic data.
    K: ArrayLike, optional
        Cholesky factor of a coefficient covariance prior (Sigma = K @ K.T),
        e.g. a GMRF smoothness prior. If given, `PrIm`/`PrKer` are assumed to
        live in the whitened coordinates beta = K @ alpha, so the basis is
        effectively Phi @ K instead of Phi. If None (default), no
        reparametrization is applied.
    """
    MIm = PrIm.shape[1]
    MKer = PrKer.shape[1]
    Phi_eff = Phi @ K if K is not None else Phi

    # --- Priors ---
    beta0 = numpyro.sample("beta0", dist.Normal(0, 1))
    betaIm = numpyro.sample("betaIm", dist.Normal(0, 1).expand([MIm]).to_event(1))
    betaKer = numpyro.sample("betaKer", dist.Normal(0, 1).expand([MKer]).to_event(1))

    # --- Latent rate ---
    lam = numpyro.deterministic(
        "lam", jnp.exp(beta0 + Phi_eff @ (PrIm @ betaIm + PrKer @ betaKer))
    )
    lam_interval = (
        jnp.matmul(W, lam) / interval_width * N[interval_id]
    )  # Average rate over each interval (divide by interval-specific width)

    # --- Likelihood ---
    with numpyro.plate("obs", len(y)):
        numpyro.sample("y", dist.Poisson(lam_interval[interval_id]), obs=y)

In [ ]:
PrKer_jax = jnp.array(PrKer)

In [ ]:
# MCMC inference with NUTS -- joint inference over betaIm and betaKer, so
# the null-space directions are constrained by the likelihood rather than
# left as pure prior draws. The median of `lam` below still uses betaIm
# only; betaKer's posterior only feeds the uncertainty band.
rng_key_mcmc = jax.random.PRNGKey(1)
mcmc = run_inference_mcmc(
    rng_key_mcmc,
    model_nullinc_bspline,
    W=W_jax,
    Phi=Phi_jax,
    PrIm=PrIm_jax,
    PrKer=PrKer_jax,
    N=N,
    interval_id=interval_id,
    interval_width=interval_width_jax,
    K=K_jax,
    y=y_jax,
)
mcmc_samples = mcmc.get_samples()
print(mcmc_samples.keys())

In [ ]:
source_exp3 = summarize_posterior_lam_betaIm_median(
    mcmc_samples,
    PrIm_jax,
    Phi_jax @ K_jax,
    np.linspace(0, 100, 100),
    "3. Null-Space Included",
)
plot_posterior(source_exp3, df_obs, df_rates)

### Experiment 4: Null-Space Prior-Uncertainty Model

Experiment 2 drops `betaKer` entirely (zero posterior uncertainty in the null space); Experiment 3 jointly fits `betaKer` alongside `betaIm` via MCMC, so its contribution to the uncertainty band is constrained by the likelihood. This experiment instead takes the Fisher-information null space at face value -- genuinely uninformative -- and shows the uncertainty that follows from that assumption alone: infer `betaIm` only (`model_reduced_bspline`, same as Experiment 2), then for each posterior draw of `betaIm` sample a fresh `betaKer` from its `Normal(0,1)` prior and propagate it through `Phi @ K` to build the band. The median `lam` line still comes from `betaIm` alone.

Comparing this against Experiment 3 isolates how much the joint MCMC fit actually narrows the null-space uncertainty relative to just trusting the prior.

In [ ]:
# MCMC inference with NUTS -- inference only over betaIm (same model as
# Experiment 2); betaKer's contribution to the uncertainty band is injected
# post-hoc from its prior below, not fit here.
rng_key_mcmc = jax.random.PRNGKey(1)
mcmc = run_inference_mcmc(
    rng_key_mcmc,
    model_reduced_bspline,
    W=W_jax,
    Phi=Phi_jax,
    PrIm=PrIm_jax,
    N=N,
    interval_id=interval_id,
    K=K_jax,
    y=y_jax,
)
mcmc_samples = mcmc.get_samples()
print(mcmc_samples.keys())

In [ ]:
source_exp4 = summarize_posterior_lam_nullspace_prior_uncertainty(
    mcmc_samples,
    PrIm_jax,
    PrKer_jax,
    Phi_jax @ K_jax,
    np.linspace(0, 100, 100),
    "4. Null-Space (Prior Uncertainty)",
    jax.random.PRNGKey(4),
    scale=0.5,
)
plot_posterior(source_exp4, df_obs, df_rates)

## Comparing All Four Experiments

In [ ]:
experiment_order = [
    "1. Naive",
    "2. Reduced Subspace",
    "3. Null-Space Included",
    "4. Null-Space (Prior Uncertainty)",
]
sources = {
    "1. Naive": source_exp1,
    "2. Reduced Subspace": source_exp2,
    "3. Null-Space Included": source_exp3,
    "4. Null-Space (Prior Uncertainty)": source_exp4,
}

comparison_charts = [
    plot_posterior(sources[name], df_obs, df_rates, title=name).properties(
        width=200, height=200
    )
    for name in experiment_order
]

alt.hconcat(*comparison_charts).resolve_scale(y="shared")